In [1]:
from sage.all import *

class SquareFree:
    """
    Sonlu cisimler üzerinde Kare Çarpansız Çarpanlara Ayırma (Square-Free Factorization, SFF)
    algoritmasının adım adım izlenebilir implementasyonu.

    Algoritma, verilen f(x) monik polinomunu f = f1^1 * f2^2 * ... * fk^k biçiminde
    yazar; her fi karecarpansiz (square-free) bir polinomdur.

    Referans: Tez, Bölüm 4 — Tek Değişkenli Polinomların Çarpanlara Ayrılması
    """

    def __init__(self, field):
        """
        Parametreler
        ------------
        field : SageMath sonlu cismi (GF(q))
            Polinomların katsayılarının ait olduğu sonlu cisim.
        """
        self.F = field
        self.q = field.order()                  # Cismin eleman sayısı q = p^m
        self.p = field.characteristic()         # Cismin karakteristiği p (asal sayı)
        self.R = PolynomialRing(self.F, 'x')    # F_q[x] tek değişkenli polinom halkası
        self.x = self.R.gen()                   # Halkanın üreteci x

        print("\n" + "="*70)
        print("SQUARE-FREE FACTORIZATION (SFF)")
        print(f"Cisim: GF({self.q}) (Karakteristik p={self.p})")
        print("="*70)

    def run_sff(self, f):
        """
        f(x) polinomu üzerinde SFF algoritmasını çalıştırır.

        Algoritma üç ana durumu ele alır:
          - Durum A : f'(x) = 0  →  f = g^p  (Frobenius tersini kullan)
          - Durum B : EBOB(f, f') = 1  →  f zaten karecarpansızdır
          - Durum C : EBOB(f, f') ≠ 1  →  katlı kökler içeriyor; döngüyle ayıkla

        Parametreler
        ------------
        f : F_q[x] polinomu
            Çarpanlara ayrılacak polinom.

        Döndürür
        --------
        list of (polinom, int)
            (çarpan, kuvvet) çiftlerinden oluşan liste.
        """
        print(f"\n>>> SFF ANALİZİ BAŞLIYOR: {f}")
        print(f"    Derece: {f.degree()}")
        print("-" * 50)

        # Adım 1: Polinomu monik yap (baş katsayıyı 1'e böl)
        # SFF yalnızca monik polinomlar üzerinde çalışır; baş katsayı dışarı alınır.
        leading = f.leading_coefficient()
        f = f / leading

        # Adım 2: Türevi hesapla
        # f'(x) = 0 ise polinom yalnızca p. kuvvetlerden oluşuyor demektir.
        df = f.derivative()
        print(f"   [1] Türev f'(x) hesaplandı: {df}")

        # ---------------------------------------------------------------
        # DURUM A: f'(x) = 0  →  f(x) = g(x)^p
        # ---------------------------------------------------------------
        # Karakteristik p'de, x^p gibi terimlerin türevi sıfır olur.
        # Bu nedenle f'= 0 ise f, p. kuvvet olan bir g(x)^p polinomuna eşittir.
        # Sonlu cisimlerde Frobenius endomorfizması terslenebilir olduğundan
        # p. kök (nth_root) işlemiyle g(x) elde edilebilir.
        if df == 0:
            print(f"   [!] TÜREV 0! Polinom tam bir {self.p}. kuvvettir (g^p).")

            # Frobenius tersini kullanarak p. kök al: g = f^(1/p)
            g = f.nth_root(self.p)
            print(f"   -> {self.p}. dereceden kök alındı: g(x) = {g}")
            print("   -> Bu kök polinom üzerinde recursive(öz yinelemeli) SFF başlatılıyor...\n")

            # g(x) üzerinde özyinelemeli SFF çağrısı
            sub_factors = self.run_sff(g)

            # Özyinelemeden dönen tüm kuvvetleri p ile çarp:
            # Eğer g = ∏ h_i^e_i ise f = g^p = ∏ h_i^(e_i * p) olur.
            final_factors = []
            for fac, power in sub_factors:
                new_power = power * self.p
                print(f"      << Recursive Dönüş: {fac} (Üs: {power} -> {new_power} oldu)")
                final_factors.append((fac, new_power))
            return final_factors

        # Adım 3: EBOB(f, f') hesapla
        # d(x) = gcd(f, f') — katlı köklerin bilgisini taşıyan yardımcı polinom
        d = gcd(f, df)
        print(f"   [2] d(x) = EBOB(f, f') hesaplandı: {d}")

        # ---------------------------------------------------------------
        # DURUM B: d = 1  →  f(x) zaten kare çarpansız (square-free)
        # ---------------------------------------------------------------
        # EBOB 1 ise f'nin f ile ortak kökü yoktur; dolayısıyla f'nin tüm
        # kökleri basittir (katlılık 1).
        if d == 1:
            print("   [SONUÇ] EBOB = 1. Polinom zaten Square-Free (Kare çarpansız).")
            return [(f, 1)]

        # ---------------------------------------------------------------
        # DURUM C: d ≠ 1  →  Katlı kökler mevcut; döngüyle çarpanlara ayır
        # ---------------------------------------------------------------
        # w(x) = f / d :
        #   Her benzersiz (kare çarpansız) çarpanın yalnızca bir kopyasını tutar.
        #   p'nin katı olmayan kuvvetlere (1, 2, 3, ...) sahip çarpanların
        #   "üssüz" çarpımıdır.
        print(f"   [3] EBOB != 1. Ayrıştırma döngüsü başlıyor...")
        w = f // d
        print(f"       w(x) = f / d = {w}")

        factors = []
        i = 1

        # Her iterasyonda i. kuvvetten gelen çarpanı ayıkla
        while w != 1:
            print(f"\n       --- Döngü {i} ---")

            # y = gcd(w, d) : (i+1) ve daha yüksek katlılıktaki çarpanları içerir
            y = gcd(w, d)
            print(f"       y = EBOB(w, d) = {y}")

            # z = w / y : tam olarak i. katlılığa sahip çarpanları verir
            z = w // y
            print(f"       z = w / y = {z}")

            if z != 1:
                # z, i. kuvvet ile çarpanlar listesine eklenir
                print(f"       *** BULUNDU! {i}. Kuvvetten Gelen Çarpan: {z}")
                factors.append((z, i))

            # Bir sonraki iterasyona geç: katlılık bilgisini güncelle
            w = y
            d = d // y
            i += 1

        # ---------------------------------------------------------------
        # p'nin katı kuvvetleri yakalama (Döngü sonrası kalan d ≠ 1 durumu)
        # ---------------------------------------------------------------
        # Ana döngü yalnızca p ile bölünemeyen kuvvetleri (1, 2, ...) işler.
        # Eğer döngü sonunda d ≠ 1 ise bu kısım p, 2p, 3p, ... gibi
        # p'nin katı olan kuvvetlere ait çarpanları barındırıyordur.
        # Bu çarpanlar d = h(x)^p biçiminde olduğundan p. köküyle geri alınır
        # ve üzerlerine özyinelemeli SFF uygulanır.
        if d != 1:
            print(f"\n   [!] Döngü bitti ama d != 1 ({d}).")
            print(f"       Bu parça p'nin katı olan kuvvetleri ({self.p}, {2*self.p}...) içerir.")

            # d = h(x)^p olduğundan Frobenius tersiyle p. kök alınır: h = d^(1/p)
            g = d.nth_root(self.p)
            print(f"   -> Kalan parçanın {self.p}. kökü alındı: {g}")
            print("   -> Bu parça için recursive SFF çağrılıyor...")

            # h(x) üzerinde özyinelemeli SFF; dönen kuvvetler p ile çarpılarak eklenir
            recursive_results = self.run_sff(g)

            for fac, power in recursive_results:
                new_power = power * self.p
                print(f"      * Recursive'den gelen çarpan eklendi: {fac} (Üs: {new_power})")
                factors.append((fac, new_power))

        return factors


def print_factors_and_verify(f, factors, R):
    """
    SFF sonucunu ekrana yazdırır ve doğrular.

    Çarpanların çarpımı orijinal monik polinom ile karşılaştırılır;
    eşleşme sağlanırsa ayrıştırmanın doğruluğu onaylanır.

    Parametreler
    ------------
    f       : Orijinal polinom (F_q[x])
    factors : (çarpan, kuvvet) çiftlerinin listesi — run_sff() çıktısı
    R       : Polinom halkası F_q[x]
    """
    print("\n" + "="*70)
    print("SQUARE-FREE FACTORIZATION SONUCU")
    print("="*70)

    # Yeniden oluşturma: tüm (çarpan^kuvvet) çarpımı hesaplanır
    reconstructed = R(1)

    print("\nÇarpan ve kuvvet listesi:")
    for factor, exponent in factors:
        print(f"   -> Çarpan: {factor}")
        print(f"      Kuvvet: {exponent}")
        reconstructed *= factor**exponent

    # Orijinal polinomu monik forma getir (baş katsayı 1)
    f_monic = f / f.leading_coefficient()

    print("\nAyrıştırma biçimi:")
    factorization_string = " * ".join(
        [f"({factor})^{exponent}" for factor, exponent in factors]
    )
    print(f"   f_monic(x) = {factorization_string}")

    # Sağlama: yeniden oluşturulan polinom orijinal ile örtüşmeli
    print("\nSağlama:")
    print(f"   Orijinal monik polinom     : {f_monic}")
    print(f"   Yeniden oluşturulan polinom: {reconstructed}")

    if reconstructed == f_monic:
        print("\nSAĞLAMA: BAŞARILI")
    else:
        print("\nSAĞLAMA: HATA")

    print("="*70)




In [2]:
# ============================================================================
# SENARYO 1 — Elle tanımlanmış polinom üzerinde SFF
# ============================================================================
# GF(3) cismi üzerinde (p=3), elle belirlenmiş bir polinom için SFF çalıştırılır.
# Bu senaryo algoritmanın deterministik davranışını gözlemlemeye yarar.

q = 3
F = GF(q)

solver = SquareFree(F)
R = solver.R
x = solver.x

f_manual = x**12 + 2*x**11 + x**7 + x**4 + 2*x**3 + x**2 + x
results = solver.run_sff(f_manual)
print_factors_and_verify(f_manual, results, R)




SQUARE-FREE FACTORIZATION (SFF)
Cisim: GF(3) (Karakteristik p=3)

>>> SFF ANALİZİ BAŞLIYOR: x^12 + 2*x^11 + x^7 + x^4 + 2*x^3 + x^2 + x
    Derece: 12
--------------------------------------------------
   [1] Türev f'(x) hesaplandı: x^10 + x^6 + x^3 + 2*x + 1
   [2] d(x) = EBOB(f, f') hesaplandı: x^7 + x^6 + x^4 + x^3 + x + 1
   [3] EBOB != 1. Ayrıştırma döngüsü başlıyor...
       w(x) = f / d = x^5 + x^4 + 2*x^3 + x

       --- Döngü 1 ---
       y = EBOB(w, d) = x + 1
       z = w / y = x^4 + 2*x^2 + x
       *** BULUNDU! 1. Kuvvetten Gelen Çarpan: x^4 + 2*x^2 + x

       --- Döngü 2 ---
       y = EBOB(w, d) = 1
       z = w / y = x + 1
       *** BULUNDU! 2. Kuvvetten Gelen Çarpan: x + 1

   [!] Döngü bitti ama d != 1 (x^6 + x^3 + 1).
       Bu parça p'nin katı olan kuvvetleri (3, 6...) içerir.
   -> Kalan parçanın 3. kökü alındı: x^2 + x + 1
   -> Bu parça için recursive SFF çağrılıyor...

>>> SFF ANALİZİ BAŞLIYOR: x^2 + x + 1
    Derece: 2
---------------------------------------

In [4]:

# ============================================================================
# SENARYO 2 — Rastgele polinom üzerinde SFF
# ============================================================================
# GF(3) cismi üzerinde, seçilen derecede rastgele üretilen bir polinom için
# SFF çalıştırılır. Her çalıştırmada farklı bir polinom üzerinde test yapılır.
# Aşağıdaki q değiştirilerek farklı cisimler için yeni senaryolar oluşturulabilir

q = 3
F = GF(q)

solver = SquareFree(F)
R = solver.R
x = solver.x

degree = 12
# Baş katsayı x^degree (monik), geri kalan terimler rastgele
# degree değeri değiştirilerek derecesi farklı rastgele polinomlar için SFF senaryosu oluşturulabilir.
f_random = x**degree + R.random_element(degree=degree - 1)
results = solver.run_sff(f_random)
print_factors_and_verify(f_random, results, R)



SQUARE-FREE FACTORIZATION (SFF)
Cisim: GF(3) (Karakteristik p=3)

>>> SFF ANALİZİ BAŞLIYOR: x^12 + 2*x^11 + 2*x^10 + x^7 + 2*x^6 + 2*x^5 + 2*x + 1
    Derece: 12
--------------------------------------------------
   [1] Türev f'(x) hesaplandı: x^10 + 2*x^9 + x^6 + x^4 + 2
   [2] d(x) = EBOB(f, f') hesaplandı: 1
   [SONUÇ] EBOB = 1. Polinom zaten Square-Free (Kare çarpansız).

SQUARE-FREE FACTORIZATION SONUCU

Çarpan ve kuvvet listesi:
   -> Çarpan: x^12 + 2*x^11 + 2*x^10 + x^7 + 2*x^6 + 2*x^5 + 2*x + 1
      Kuvvet: 1

Ayrıştırma biçimi:
   f_monic(x) = (x^12 + 2*x^11 + 2*x^10 + x^7 + 2*x^6 + 2*x^5 + 2*x + 1)^1

Sağlama:
   Orijinal monik polinom     : x^12 + 2*x^11 + 2*x^10 + x^7 + 2*x^6 + 2*x^5 + 2*x + 1
   Yeniden oluşturulan polinom: x^12 + 2*x^11 + 2*x^10 + x^7 + 2*x^6 + 2*x^5 + 2*x + 1

SAĞLAMA: BAŞARILI
